# Euro 2024 TOP-15: clean pipeline

Порядок работы в этом ноутбуке:
1. Скачать и сохранить локально 15 матчей (`events` + `three-sixty`).
2. Задать вручную словарь по матчам: команды, цвета, `ref_by_period`.
3. Прогнать стандартизацию и собрать цепочки.
4. Визуально проверить payload/координаты/360.
5. Только после этого считать доп. признаки.


In [ ]:
from pathlib import Path
import json
import requests
import pandas as pd
import importlib
# Импортируем и принудительно перезагружаем локальные модули
from statsbomb_toolkit.sber_exports import pipeline as _pipeline, chaining, prompting, metrics, viz as _viz, preprocessing as _preprocessing
pipeline = importlib.reload(_pipeline)
viz = importlib.reload(_viz)
preprocessing = importlib.reload(_preprocessing)
print('pipeline module:', getattr(pipeline, '__file__', 'unknown'))
print('has bad_ids_breakdown:', hasattr(pipeline, 'bad_ids_breakdown'))


In [ ]:
# --- paths ---
ROOT = Path('/Users/angelina23/Documents/ml-ami')
DATA_DIR = ROOT / 'data' / 'euro2024_all'
EVENTS_DIR = DATA_DIR / 'events'
SB360_DIR = DATA_DIR / 'three-sixty'
OUT_DIR = ROOT / 'outputs' / 'euro2024_all'
for d in [DATA_DIR, EVENTS_DIR, SB360_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print('DATA_DIR:', DATA_DIR)
print('OUT_DIR :', OUT_DIR)


In [ ]:
# --- competition setup + fixed TOP-15 speak list ---
EURO_COMPETITION_ID = 55
EURO_2024_SEASON_ID = 282  # UEFA Euro 2024 in StatsBomb Open Data

OPENING_MATCH_ID = '3930158'
QF_SF_FINAL_IDS = ['3942349', '3942226', '3942382', '3942227', '3942752', '3942819', '3943043']
R16_TOP7_IDS = ['3941018', '3941017', '3941022', '3941021', '3940983', '3940878', '3941019']
TOP15_MATCH_IDS = [OPENING_MATCH_ID] + QF_SF_FINAL_IDS + R16_TOP7_IDS
TOP15_SET = set(TOP15_MATCH_IDS)
print('TOP15 count:', len(TOP15_MATCH_IDS))
print(TOP15_MATCH_IDS)


In [ ]:
# --- download helpers (ALL Euro 2024 + TOP15 mark) ---
BASE_RAW = 'https://raw.githubusercontent.com/statsbomb/open-data/master/data'


def _download_json(url: str, dst: Path, timeout=30):
    if dst.exists():
        return 'exists'
    r = requests.get(url, timeout=timeout)
    if r.status_code != 200:
        return f'http_{r.status_code}'
    dst.write_text(r.text, encoding='utf-8')
    return 'ok'


def fetch_euro2024_matches(competition_id=EURO_COMPETITION_ID, season_id=EURO_2024_SEASON_ID):
    url = f'{BASE_RAW}/matches/{competition_id}/{season_id}.json'
    r = requests.get(url, timeout=40)
    r.raise_for_status()
    data = r.json()
    rows = []
    for m in data:
        mid = str(m.get('match_id'))
        home = ((m.get('home_team') or {}).get('home_team_name')
                or (m.get('home_team') or {}).get('name'))
        away = ((m.get('away_team') or {}).get('away_team_name')
                or (m.get('away_team') or {}).get('name'))
        stage = ((m.get('competition_stage') or {}).get('name'))
        mdate = m.get('match_date')
        rows.append({
            'match_id': mid,
            'home_team': home,
            'away_team': away,
            'stage': stage,
            'match_date': mdate,
            'to_speak': int(mid in TOP15_SET),
        })
    df = pd.DataFrame(rows).drop_duplicates(subset=['match_id']).sort_values(['match_date', 'match_id']).reset_index(drop=True)
    return df


def download_matches_local(match_ids):
    rows = []
    for mid in match_ids:
        ev_url = f'{BASE_RAW}/events/{mid}.json'
        s3_url = f'{BASE_RAW}/three-sixty/{mid}.json'
        ev_dst = EVENTS_DIR / f'{mid}.json'
        s3_dst = SB360_DIR / f'{mid}.json'
        ev_status = _download_json(ev_url, ev_dst)
        s3_status = _download_json(s3_url, s3_dst)
        rows.append({'match_id': mid, 'events': ev_status, 'three_sixty': s3_status, 'to_speak': int(mid in TOP15_SET)})
    return pd.DataFrame(rows)


In [ ]:
# --- run download: ALL Euro 2024 + TOP15 subset ---
df_matches = fetch_euro2024_matches()
ALL_EURO_MATCH_IDS = df_matches['match_id'].astype(str).tolist()
print('Euro 2024 matches total:', len(ALL_EURO_MATCH_IDS))
print('TOP15 in list:', int(df_matches['to_speak'].sum()))

df_dl = download_matches_local(ALL_EURO_MATCH_IDS)
display(df_dl.head(10))

# merge manifest
manifest = df_matches.merge(df_dl, on=['match_id', 'to_speak'], how='left')
manifest = manifest.sort_values(['match_date', 'match_id']).reset_index(drop=True)
manifest_path = OUT_DIR / 'euro2024_manifest_all.csv'
manifest.to_csv(manifest_path, index=False)
print('saved manifest:', manifest_path)

manifest_top15 = manifest[manifest['to_speak'] == 1].copy()
manifest_top15_path = OUT_DIR / 'euro2024_manifest_to_speak_top15.csv'
manifest_top15.to_csv(manifest_top15_path, index=False)
print('saved TOP15 manifest:', manifest_top15_path)

print('events local:', sum((EVENTS_DIR / f'{mid}.json').exists() for mid in ALL_EURO_MATCH_IDS), '/', len(ALL_EURO_MATCH_IDS))
print('sb360  local:', sum((SB360_DIR / f'{mid}.json').exists() for mid in ALL_EURO_MATCH_IDS), '/', len(ALL_EURO_MATCH_IDS))


In [ ]:
# --- load helpers ---
def load_events(mid: str):
    return json.loads((EVENTS_DIR / f'{mid}.json').read_text(encoding='utf-8'))
def load_sb360(mid: str):
    p = SB360_DIR / f'{mid}.json'
    return json.loads(p.read_text(encoding='utf-8')) if p.exists() else []


## Ручной словарь матчей

Здесь ты задаёшь:
- `teams` (для контроля),
- `team_colors`,
- `ref_by_period` (ключевая часть стандартизации).


In [ ]:
TEAM_COLORS_DEFAULT = {
    'Scotland': '#1f77b4', 'Germany': '#111111', 'Spain': '#c62828', 'England': '#f2f2f2',
    'France': '#1e3a8a', 'Italy': '#1d4ed8', 'Portugal': '#b91c1c', 'Netherlands': '#ea580c',
    'Belgium': '#7f1d1d', 'Croatia': '#dc2626', 'Switzerland': '#ef4444', 'Denmark': '#b91c1c',
    'Austria': '#dc2626', 'Turkey': '#b91c1c', 'Georgia': '#2563eb', 'Romania': '#facc15',
    'Slovenia': '#16a34a', 'Slovakia': '#2563eb', 'Serbia': '#b91c1c', 'Ukraine': '#2563eb',
    'Poland': '#ef4444', 'Albania': '#b91c1c', 'Hungary': '#b91c1c', 'Czechia': '#dc2626',
    'Czech Republic': '#dc2626',
}


def _infer_teams_from_events(events_raw):
    teams = []
    for ev in events_raw:
        t = ((ev.get('team') or {}).get('name'))
        if t and t not in teams:
            teams.append(t)
        if len(teams) >= 2:
            break
    if len(teams) < 2:
        all_teams = []
        for ev in events_raw:
            t = ((ev.get('team') or {}).get('name'))
            if t and t not in all_teams:
                all_teams.append(t)
        teams = all_teams[:2]
    return teams


def _auto_ref_by_period(teams):
    # Автозаготовка. Для части матчей вручную поправим.
    if len(teams) >= 2:
        return {1: teams[0], 2: teams[1]}
    if len(teams) == 1:
        return {1: teams[0], 2: teams[0]}
    return {1: 'UNKNOWN', 2: 'UNKNOWN'}


def build_match_configs_auto(match_ids):
    cfg = {}
    for mid in match_ids:
        p = EVENTS_DIR / f'{mid}.json'
        if not p.exists():
            continue
        events_raw = json.loads(p.read_text(encoding='utf-8'))
        teams = _infer_teams_from_events(events_raw)
        team_colors = {t: TEAM_COLORS_DEFAULT.get(t, '#9ca3af') for t in teams}
        cfg[int(mid)] = {
            'teams': teams,
            'team_colors': team_colors,
            'ref_by_period': _auto_ref_by_period(teams),
            'to_speak': int(str(mid) in TOP15_SET),
        }
    return cfg


MATCH_CONFIGS = build_match_configs_auto(ALL_EURO_MATCH_IDS)

# Жёсткие ручные фиксы (дополняй по мере визуальной проверки)
MANUAL_REF_OVERRIDES = {
    3930158: {1: 'Scotland', 2: 'Germany'},
}
for mid, ref in MANUAL_REF_OVERRIDES.items():
    if mid in MATCH_CONFIGS:
        MATCH_CONFIGS[mid]['ref_by_period'] = ref

print('configured matches:', len(MATCH_CONFIGS))
print('configured to_speak:', sum(v.get('to_speak', 0) for v in MATCH_CONFIGS.values()))

# Ручные фиксы по jersey
MANUAL_JERSEY_OVERRIDES = {
    3930158: {
        'by_id': {},
        'by_name': {},
    },
}
for mid, ov in MANUAL_JERSEY_OVERRIDES.items():
    if mid in MATCH_CONFIGS:
        MATCH_CONFIGS[mid]['jersey_overrides_by_id'] = ov.get('by_id', {})
        MATCH_CONFIGS[mid]['jersey_overrides_by_name'] = ov.get('by_name', {})



In [ ]:
# --- ref-team helpers (manual control by half) ---
def build_ref_table(match_configs):
    rows = []
    for mid in sorted(match_configs.keys()):
        cfg = match_configs[mid]
        teams = cfg.get('teams', [])
        ref = cfg.get('ref_by_period', {})
        rows.append({
            'match_id': mid,
            'team_1': teams[0] if len(teams) > 0 else None,
            'team_2': teams[1] if len(teams) > 1 else None,
            'ref_p1': ref.get(1),
            'ref_p2': ref.get(2),
            'to_speak': int(cfg.get('to_speak', 0)),
            'team_colors': cfg.get('team_colors', {}),
            'jersey_fix_by_id_n': len(cfg.get('jersey_overrides_by_id', {})),
            'jersey_fix_by_name_n': len(cfg.get('jersey_overrides_by_name', {})),
        })
    return pd.DataFrame(rows).sort_values(['to_speak', 'match_id'], ascending=[False, True]).reset_index(drop=True)


def set_ref_for_match(match_id: int, ref_p1: str, ref_p2: str):
    if match_id not in MATCH_CONFIGS:
        raise KeyError(f'match_id={match_id} not in MATCH_CONFIGS')
    MATCH_CONFIGS[match_id]['ref_by_period'] = {1: ref_p1, 2: ref_p2}


def apply_ref_overrides(overrides: dict):
    for mid, refmap in overrides.items():
        if int(mid) in MATCH_CONFIGS:
            MATCH_CONFIGS[int(mid)]['ref_by_period'] = {1: refmap.get(1), 2: refmap.get(2)}


def save_ref_overrides_json(path):
    path = Path(path)
    payload = {str(k): v.get('ref_by_period', {}) for k, v in MATCH_CONFIGS.items()}
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    return path


def load_ref_overrides_json(path):
    path = Path(path)
    data = json.loads(path.read_text(encoding='utf-8'))
    for k, refmap in data.items():
        mid = int(k)
        if mid in MATCH_CONFIGS:
            MATCH_CONFIGS[mid]['ref_by_period'] = {1: refmap.get('1', refmap.get(1)), 2: refmap.get('2', refmap.get(2))}


def build_fixed_match_dicts_from_configs(match_configs):
    ref_fixed = {}
    colors_fixed = {}
    jersey_fixed = {}
    for mid, cfg in sorted(match_configs.items()):
        ref_fixed[mid] = dict(cfg.get('ref_by_period', {}))
        colors_fixed[mid] = dict(cfg.get('team_colors', {}))
        jersey_fixed[mid] = {
            'by_id': dict(cfg.get('jersey_overrides_by_id', {})),
            'by_name': dict(cfg.get('jersey_overrides_by_name', {})),
        }
    return ref_fixed, colors_fixed, jersey_fixed


def apply_fixed_match_dicts(ref_fixed=None, colors_fixed=None, jersey_fixed=None):
    ref_fixed = ref_fixed or {}
    colors_fixed = colors_fixed or {}
    jersey_fixed = jersey_fixed or {}
    for mid, refmap in ref_fixed.items():
        if int(mid) in MATCH_CONFIGS:
            MATCH_CONFIGS[int(mid)]['ref_by_period'] = {1: refmap.get(1, refmap.get('1')), 2: refmap.get(2, refmap.get('2'))}
    for mid, cmap in colors_fixed.items():
        if int(mid) in MATCH_CONFIGS:
            MATCH_CONFIGS[int(mid)]['team_colors'] = dict(cmap)
    for mid, ov in jersey_fixed.items():
        if int(mid) in MATCH_CONFIGS:
            MATCH_CONFIGS[int(mid)]['jersey_overrides_by_id'] = dict((ov or {}).get('by_id', {}))
            MATCH_CONFIGS[int(mid)]['jersey_overrides_by_name'] = dict((ov or {}).get('by_name', {}))


REF_BY_PERIOD_FIXED, TEAM_COLORS_FIXED_BY_MATCH, JERSEY_FIXED_BY_MATCH = build_fixed_match_dicts_from_configs(MATCH_CONFIGS)

display(build_ref_table(MATCH_CONFIGS))


In [ ]:
# --- detailed bad_id stats for one match or all runs ---
def bad_id_stats_for_run(run_obj):
    events_raw = run_obj['events_raw']
    bad_ids = run_obj['bad_ids']
    reasons = run_obj['bad_meta'].get('reasons', {})
    total = len(events_raw)
    bad_n = len(bad_ids)
    rows = []
    for ev in events_raw:
        eid = ev.get('id')
        if eid in bad_ids:
            rows.append({
                'event_id': eid,
                'period': ev.get('period'),
                'timestamp': ev.get('timestamp'),
                'type': ((ev.get('type') or {}).get('name')),
                'team': ((ev.get('team') or {}).get('name')),
                'player': ((ev.get('player') or {}).get('name')),
            })
    df_bad = pd.DataFrame(rows)
    by_type = df_bad.groupby('type').size().sort_values(ascending=False).rename('n').reset_index() if not df_bad.empty else pd.DataFrame(columns=['type','n'])
    by_period = df_bad.groupby('period').size().sort_values(ascending=False).rename('n').reset_index() if not df_bad.empty else pd.DataFrame(columns=['period','n'])
    print('match_id:', run_obj['match_id'])
    print('bad_ids:', bad_n, '/', total, f"({round(100*bad_n/max(total,1),2)}%)")
    print('reasons:', reasons)
    print('\nby period:')
    display(by_period)
    print('\nby type:')
    display(by_type)
    return df_bad, by_type, by_period
def bad_id_stats_all_runs(all_runs):
    rows = []
    for mid, run_obj in all_runs.items():
        total = len(run_obj['events_raw'])
        bad_n = len(run_obj['bad_ids'])
        rs = run_obj['bad_meta'].get('reasons', {})
        rows.append({
            'match_id': mid,
            'events_n': total,
            'bad_ids_n': bad_n,
            'bad_share_pct': round(100*bad_n/max(total,1),2),
            'bad_non_play': rs.get('non_play'),
            'bad_low_signal': rs.get('low_signal'),
            'bad_missing_360': rs.get('missing_360'),
            'bad_tiny_ff': rs.get('tiny_freeze_frame'),
            'bad_actor_mismatch_dist': rs.get('actor_mismatch_dist'),
        })
    return pd.DataFrame(rows).sort_values('match_id').reset_index(drop=True)


In [ ]:
# --- bad events / bad 360 filters ---
# Используем функцию из sber_exports:
# bad_ids = pipeline.build_bad_ids(events_raw, sb360_raw, min_freeze_players=3)


In [ ]:
# --- compatibility helper for bad_ids breakdown ---
def get_bad_meta(events_raw, sb360_raw, **kwargs):
    if hasattr(pipeline, 'bad_ids_breakdown'):
        return pipeline.bad_ids_breakdown(events_raw, sb360_raw, **kwargs)
    # fallback для старой версии модуля
    bad_ids = pipeline.build_bad_ids(events_raw, sb360_raw, **kwargs)
    return {
        'bad_ids': bad_ids,
        'reasons': {
            'non_play': None,
            'low_signal': None,
            'missing_360': None,
            'tiny_freeze_frame': None,
            'actor_mismatch_dist': None,
        },
    }


In [ ]:
# --- standardize one match end-to-end (NO CHAINS) ---
def run_match_standardize_only(
    match_id: int,
    min_freeze_players=3,
    include_non_play_bad=False,
    include_low_signal_bad=False,
    include_missing_360_bad=False,
    include_tiny_freeze_bad=False,
    include_actor_mismatch_bad=True,
    max_actor_event_dist=15.0,
):
    if match_id not in MATCH_CONFIGS:
        raise KeyError(f'match_id={match_id} not in MATCH_CONFIGS')
    events_raw = load_events(str(match_id))
    sb360_raw = load_sb360(str(match_id))
    cfg = MATCH_CONFIGS[match_id]
    bad_meta = get_bad_meta(
        events_raw,
        sb360_raw,
        min_freeze_players=min_freeze_players,
        include_non_play=include_non_play_bad,
        include_low_signal=include_low_signal_bad,
        include_missing_360=include_missing_360_bad,
        include_tiny_freeze=include_tiny_freeze_bad,
        include_actor_mismatch=include_actor_mismatch_bad,
        max_actor_event_dist=max_actor_event_dist,
    )
    bad_ids = bad_meta['bad_ids']
    # 1) half-aware standardization (events + 360)
    events_std, sb360_std, flip_map = preprocessing.standardize_events_and_360_by_half(
        events_raw,
        sb360_raw,
        cfg['ref_by_period'],
    )
    # 2) indexes
    sb360_by_id = {}
    for s in sb360_std:
        eid = s.get('event_uuid') or s.get('id')
        if eid:
            sb360_by_id[eid] = s
    # 3) clean events for LLM (drop noisy fields) + surname-only player names
    events_std_clean = [preprocessing.clean_event_for_llm(e) for e in events_std]
    events_std_clean = [preprocessing.to_surname_event(e) for e in events_std_clean]
    events_std_clean_by_id = {e.get('id'): e for e in events_std_clean if e.get('id')}
    # 4) standardized objects for model/debug: bad_ids => sb360_json=None
    llm_items = []
    for ev in events_std:
        eid = ev.get('id')
        ev_clean = events_std_clean_by_id.get(eid)
        if not ev_clean:
            continue
        sb = None if eid in bad_ids else sb360_by_id.get(eid)
        llm_items.append({'event_id': eid, 'event_json': ev_clean, 'sb360_json': sb})
    # 5) context for full-style visualization in half_switch_test logic (from RAW)
    jersey_by_id, jersey_by_name = viz.build_jersey_maps(
        events_raw,
        fixed_by_id=cfg.get('jersey_overrides_by_id', {}),
        fixed_by_name=cfg.get('jersey_overrides_by_name', {}),
    )
    df_events_viz = viz.make_viz_events_df(events_raw, jersey_by_id, jersey_by_name)
    frames_by_id_viz, visible_by_id_viz = viz.make_360_indexes(sb360_raw)
    return {
        'match_id': match_id,
        'cfg': cfg,
        'to_speak': int(str(match_id) in TOP15_SET),
        'events_raw_n': len(events_raw),
        'sb360_raw_n': len(sb360_raw),
        'events_raw': events_raw,
        'sb360_raw': sb360_raw,
        'events_std': events_std,
        'events_std_clean': events_std_clean,
        'events_std_clean_by_id': events_std_clean_by_id,
        'sb360_std': sb360_std,
        'sb360_by_id': sb360_by_id,
        'flip_map': flip_map,
        'bad_ids': bad_ids,
        'bad_meta': bad_meta,
        'llm_items': llm_items,
        # backward-compatible alias:
        'viz_items': llm_items,
        'df_events_viz': df_events_viz,
        'frames_by_id_viz': frames_by_id_viz,
        'visible_by_id_viz': visible_by_id_viz,
    }



In [ ]:
# --- diagnostics ---
def summarize_standardized(run_obj):
    events_std = run_obj['events_std']
    events_std_clean = run_obj['events_std_clean']
    llm_items = run_obj['llm_items']
    bad_ids = run_obj['bad_ids']
    bad_meta = run_obj['bad_meta']
    print('match_id:', run_obj['match_id'])
    print('events_raw_n :', run_obj['events_raw_n'])
    print('sb360_raw_n  :', run_obj['sb360_raw_n'])
    print('events_std   :', len(events_std))
    print('events_clean :', len(events_std_clean))
    print('llm_items    :', len(llm_items))
    print('bad_ids      :', len(bad_ids))
    print('bad reasons  :', bad_meta.get('reasons', {}))
    # share of disabled 360 in llm items
    no360 = sum(1 for x in llm_items if x.get('sb360_json') is None)
    print('llm_items with sb360=None:', no360, '/', len(llm_items), f"({round(100*no360/max(len(llm_items),1),2)}%)")
    # orientation table
    orient_rows = []
    for ev in events_std:
        team = ((ev.get('team') or {}).get('name'))
        period = int(ev.get('period') or 1)
        own, opp, _ = preprocessing._own_opp_goal_x_for_event(ev, run_obj['cfg']['ref_by_period'])
        orient_rows.append({'period': period, 'team': team, 'own_goal_x': own, 'opp_goal_x': opp})
    df_or = pd.DataFrame(orient_rows).drop_duplicates().sort_values(['period', 'team']).reset_index(drop=True)
    display(df_or)
    # quick clean sanity: possession fields dropped, names shortened
    if events_std and events_std_clean:
        raw0 = events_std[0]
        cln0 = events_std_clean[0]
        print('clean sanity -> has possession:', 'possession' in cln0, 'possession_team' in cln0)
        print('clean sanity -> sample player raw/clean:', ((raw0.get('player') or {}).get('name')), '=>', ((cln0.get('player') or {}).get('name')))


In [ ]:
# --- visualization (full-style, half_switch logic, NO CHAINS) ---
import matplotlib.pyplot as plt
from IPython.display import display
import importlib
import numpy as np
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass
importlib.reload(viz)
def _has_actor_candidate(df_events_viz, frames_by_id_viz, eid, allow_360=True):
    row = df_events_viz.loc[df_events_viz['id'] == eid]
    if row.empty:
        return False
    r = row.iloc[0]
    # event location available
    ex = r.get('event_x', np.nan)
    ey = r.get('event_y', np.nan)
    if pd.notna(ex) and pd.notna(ey):
        return True
    if not allow_360:
        return False
    # actor from 360
    fr = (frames_by_id_viz or {}).get(eid)
    if fr is None or len(fr) == 0:
        return False
    if 'actor' in fr.columns:
        fr_actor = fr[fr['actor'] == True]
        if len(fr_actor) > 0:
            ax = fr_actor.iloc[0].get('x', np.nan)
            ay = fr_actor.iloc[0].get('y', np.nan)
            if pd.notna(ax) and pd.notna(ay):
                return True
    return False
def _pick_event_ids(df_events_viz, frames_by_id_viz, bad_ids, period, n=8, with360=True):
    d = df_events_viz[df_events_viz['period'] == period].sort_values('index')
    ids = []
    for eid in d['id'].tolist():
        is_bad = eid in bad_ids
        if with360 and is_bad:
            continue
        if (not with360) and (not is_bad):
            continue
        # avoid service events in showcase
        type_name = d.loc[d['id'] == eid, 'type_name'].iloc[0]
        if type_name in {'Starting XI', 'Half Start', 'Half End'}:
            continue
        # must have drawable actor candidate
        if not _has_actor_candidate(df_events_viz, frames_by_id_viz, eid, allow_360=with360):
            continue
        ids.append(eid)
        if len(ids) >= n:
            break
    return ids
def show_standardized_visuals(run_obj, n_with360=8, n_without360=8):
    cfg = run_obj['cfg']
    bad_ids = run_obj['bad_ids']
    df_events_viz = run_obj['df_events_viz']
    frames_by_id_viz = run_obj['frames_by_id_viz']
    visible_by_id_viz = run_obj['visible_by_id_viz']
    teams = cfg.get('teams', [])
    colors = cfg.get('team_colors', {})
    ref_map = cfg.get('ref_by_period', {})
    ref_p1 = ref_map.get(1, teams[0] if teams else None)
    def _draw(eids, title, disable_360=False):
        print(title, '| n=', len(eids))
        if eids:
            print('event_ids:', eids)
        if not eids:
            print('  (пусто)')
            return
        for eid in eids:
            local_frames = {} if disable_360 else frames_by_id_viz
            local_visible = {} if disable_360 else visible_by_id_viz
            fig, ax = viz.draw_event_keep_style_half_switch_full(
                df_events_viz,
                eid,
                local_frames,
                local_visible,
                reference_team=ref_p1,
                ref_by_period=ref_map,
                team_colors=colors,
                show_badge=True,
                show_header=True,
                show_ref=False,
            )
            display(fig)
            plt.close(fig)
    p1_with = _pick_event_ids(df_events_viz, frames_by_id_viz, bad_ids, period=1, n=n_with360, with360=True)
    p2_with = _pick_event_ids(df_events_viz, frames_by_id_viz, bad_ids, period=2, n=n_with360, with360=True)
    p1_no = _pick_event_ids(df_events_viz, frames_by_id_viz, bad_ids, period=1, n=n_without360, with360=False)
    p2_no = _pick_event_ids(df_events_viz, frames_by_id_viz, bad_ids, period=2, n=n_without360, with360=False)
    _draw(p1_with, '=== PERIOD 1 | with 360 ===', disable_360=False)
    _draw(p2_with, '=== PERIOD 2 | with 360 ===', disable_360=False)
    _draw(p1_no, '=== PERIOD 1 | 360 disabled by bad_ids ===', disable_360=True)
    _draw(p2_no, '=== PERIOD 2 | 360 disabled by bad_ids ===', disable_360=True)


In [ ]:
# --- run one match: executable example (NO CHAINS) ---
match_id = 3930158
run_obj = run_match_standardize_only(
    match_id,
    min_freeze_players=3,
    include_non_play_bad=False,
    include_low_signal_bad=False,
    include_missing_360_bad=False,
    include_tiny_freeze_bad=False,
    include_actor_mismatch_bad=True,
    max_actor_event_dist=15.0,
)
summarize_standardized(run_obj)
show_standardized_visuals(run_obj, n_with360=8, n_without360=8)


In [ ]:
# --- quick visual run: executable example (NO CHAINS) ---
def quick_visual_run(match_id=3930158):
    run_obj = run_match_standardize_only(
        match_id,
        min_freeze_players=3,
        include_non_play_bad=False,
        include_low_signal_bad=False,
        include_missing_360_bad=False,
        include_tiny_freeze_bad=False,
        include_actor_mismatch_bad=True,
        max_actor_event_dist=15.0,
    )
    summarize_standardized(run_obj)
    show_standardized_visuals(run_obj, n_with360=8, n_without360=8)
    return run_obj
run_obj_q = quick_visual_run(3930158)


In [ ]:
# --- standardized previews (NO CHAINS) ---
import json
def show_standardized_previews(run_obj, k=6):
    items = run_obj['viz_items']
    print('items total:', len(items))
    with360 = [x for x in items if x.get('sb360_json') is not None]
    no360 = [x for x in items if x.get('sb360_json') is None]
    print('with 360   :', len(with360))
    print('without 360:', len(no360))
    print('\n=== SAMPLE WITH 360 ===')
    for it in with360[:k]:
        ej = it.get('event_json') or {}
        print((ej.get('timestamp'), (ej.get('type') or {}).get('name'), (ej.get('player') or {}).get('name'), it.get('event_id')))
    print('\n=== SAMPLE WITHOUT 360 ===')
    for it in no360[:k]:
        ej = it.get('event_json') or {}
        print((ej.get('timestamp'), (ej.get('type') or {}).get('name'), (ej.get('player') or {}).get('name'), it.get('event_id')))
    if items:
        demo = {
            'event_id': items[0].get('event_id'),
            'event_json': items[0].get('event_json'),
            'sb360_json': items[0].get('sb360_json'),
        }
        print('\n=== FIRST ITEM JSON PREVIEW ===')
        print(json.dumps(demo, ensure_ascii=False, indent=2)[:4000])


In [ ]:
# --- standardized previews: executable example ---
show_standardized_previews(run_obj, k=6)


In [ ]:
# --- run all configured matches (ALL EURO 2024, NO CHAINS) ---
def run_all_matches_standardize_only(
    only_to_speak=False,
    min_freeze_players=3,
    include_non_play_bad=False,
    include_low_signal_bad=False,
    include_missing_360_bad=False,
    include_tiny_freeze_bad=False,
    include_actor_mismatch_bad=True,
    max_actor_event_dist=15.0,
):
    all_runs = {}
    rows = []
    mids = sorted(MATCH_CONFIGS.keys())
    for mid_int in mids:
        cfg = MATCH_CONFIGS[mid_int]
        to_speak = int(cfg.get('to_speak', 0))
        if only_to_speak and to_speak != 1:
            continue

        ev_path = EVENTS_DIR / f'{mid_int}.json'
        if not ev_path.exists():
            rows.append({'match_id': mid_int, 'to_speak': to_speak, 'status': 'skip_no_events_file'})
            continue

        run_obj = run_match_standardize_only(
            mid_int,
            min_freeze_players=min_freeze_players,
            include_non_play_bad=include_non_play_bad,
            include_low_signal_bad=include_low_signal_bad,
            include_missing_360_bad=include_missing_360_bad,
            include_tiny_freeze_bad=include_tiny_freeze_bad,
            include_actor_mismatch_bad=include_actor_mismatch_bad,
            max_actor_event_dist=max_actor_event_dist,
        )
        reasons = run_obj['bad_meta']['reasons']
        rows.append({
            'match_id': mid_int,
            'to_speak': to_speak,
            'status': 'ok',
            'events_std_n': len(run_obj['events_std']),
            'viz_items_n': len(run_obj['viz_items']),
            'bad_ids_n': len(run_obj['bad_ids']),
            'bad_non_play': reasons['non_play'],
            'bad_low_signal': reasons['low_signal'],
            'bad_missing_360': reasons['missing_360'],
            'bad_tiny_ff': reasons['tiny_freeze_frame'],
            'bad_actor_mismatch_dist': reasons['actor_mismatch_dist'],
        })
        all_runs[mid_int] = run_obj

    df = pd.DataFrame(rows).sort_values(['to_speak', 'match_id'], ascending=[False, True]).reset_index(drop=True)
    return all_runs, df


In [ ]:
# --- run all configured matches: executable example ---
all_runs, df_all = run_all_matches_standardize_only(
    only_to_speak=False,  # True -> только TOP15
    min_freeze_players=3,
    include_non_play_bad=False,
    include_low_signal_bad=False,
    include_missing_360_bad=False,
    include_tiny_freeze_bad=False,
    include_actor_mismatch_bad=True,
    max_actor_event_dist=15.0,
)
display(df_all)

all_summary_path = OUT_DIR / 'euro2024_all_standardize_summary.csv'
df_all.to_csv(all_summary_path, index=False)
print('saved:', all_summary_path)

# отдельная таблица только для TOP15 to_speak
if 'to_speak' in df_all.columns:
    df_top15 = df_all[df_all['to_speak'] == 1].copy()
else:
    df_top15 = df_all[df_all['match_id'].astype(str).isin(TOP15_SET)].copy()

top15_summary_path = OUT_DIR / 'euro2024_top15_to_speak_standardize_summary.csv'
df_top15.to_csv(top15_summary_path, index=False)
print('saved:', top15_summary_path)
print('TOP15 processed rows:', len(df_top15))


In [ ]:
# --- quick visuals (optionally only TOP15) ---
def show_all_matches_quick_visuals(all_runs, only_to_speak=True, n_with360=3, n_without360=3):
    mids = sorted(all_runs.keys())
    for mid_int in mids:
        run_obj = all_runs.get(mid_int)
        if run_obj is None:
            continue
        if only_to_speak and int(run_obj.get('to_speak', 0)) != 1:
            continue

        cfg = run_obj['cfg']
        teams = cfg.get('teams', [])
        ref = cfg.get('ref_by_period', {})
        bad_n = len(run_obj.get('bad_ids', set()))
        ev_n = len(run_obj.get('events_raw', []))
        print('\n' + '='*100)
        print(f"MATCH {mid_int} | to_speak={run_obj.get('to_speak', 0)}")
        print(f'teams: {teams}')
        print(f'ref_by_period: {ref}')
        print(f'bad_ids: {bad_n}/{ev_n} ({round(100*bad_n/max(ev_n,1),2)}%)')
        show_standardized_visuals(run_obj, n_with360=n_with360, n_without360=n_without360)

# execute by default: показываем только TOP15
show_all_matches_quick_visuals(all_runs, only_to_speak=True, n_with360=3, n_without360=3)


In [ ]:
# --- save chains to files (optional export) ---
def build_and_save_chains_for_run(run_obj, out_dir=OUT_DIR):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    match_id = run_obj['match_id']
    ref_by_period = run_obj['cfg']['ref_by_period']
    bad_ids = run_obj['bad_ids']
    events_std = run_obj['events_std']
    sb360_std = run_obj['sb360_std']
    _, neighbors = chaining.build_related_graph(events_std)
    chains = chaining.build_nonoverlap_chains(events_std, neighbors)
    events_by_id, events_clean_by_id, sb360_by_id = pipeline.build_maps(events_std, sb360_std)
    payloads = pipeline.build_chain_payloads(
        chains,
        events_by_id=events_by_id,
        events_clean_by_id=events_clean_by_id,
        sb360_by_id=sb360_by_id,
        bad_ids=set(bad_ids),
        ref_by_period=ref_by_period,
    )
    p_ids = out_dir / f'{match_id}_chains_ids.json'
    p_payloads = out_dir / f'{match_id}_chains_payloads.jsonl'
    p_ids.write_text(json.dumps(chains, ensure_ascii=False, indent=2), encoding='utf-8')
    with p_payloads.open('w', encoding='utf-8') as f:
        for row in payloads:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    print('saved:', p_ids)
    print('saved:', p_payloads)
    print('chains:', len(chains), '| payloads:', len(payloads))
    return {'chains': chains, 'payloads': payloads, 'paths': {'ids': p_ids, 'payloads': p_payloads}}
def save_chains_for_all_runs(all_runs, out_dir=OUT_DIR / 'chains_top15'):
    out = {}
    for mid, run_obj in all_runs.items():
        print('\n' + '='*80)
        print('match', mid)
        out[mid] = build_and_save_chains_for_run(run_obj, out_dir=out_dir)
    return out
# Примеры:
# chains_one = build_and_save_chains_for_run(run_obj)
# chains_all = save_chains_for_all_runs(all_runs)


In [ ]:
# --- sanity check: clean_recursive + surname ---
def show_cleaning_sanity(match_id=3930158, event_pos=0):
    events_raw = load_events(str(match_id))
    ev_raw = events_raw[event_pos]
    ev_clean = preprocessing.clean_event_for_llm(ev_raw)
    ev_surname = preprocessing.to_surname_event(ev_clean)
    print('type:', preprocessing.get_type_name(ev_raw), '| id:', ev_raw.get('id'))
    print('raw player      :', ((ev_raw.get('player') or {}).get('name')))
    print('clean player    :', ((ev_clean.get('player') or {}).get('name')))
    print('surname player  :', ((ev_surname.get('player') or {}).get('name')))
    raw_rec = (((ev_raw.get('pass') or {}).get('recipient') or {}).get('name'))
    clean_rec = (((ev_clean.get('pass') or {}).get('recipient') or {}).get('name'))
    sur_rec = (((ev_surname.get('pass') or {}).get('recipient') or {}).get('name'))
    print('raw recipient   :', raw_rec)
    print('clean recipient :', clean_rec)
    print('surname recip.  :', sur_rec)
    # пример: в clean нет possession-полей
    print('has possession in clean:', 'possession' in ev_clean, 'possession_team' in ev_clean)
# пример:
# show_cleaning_sanity(match_id=3930158, event_pos=100)


## Доп. признаки временно отключены

Сейчас блок отключён по задаче: работаем только с
- стандартизацией,
- фильтрацией bad 360,
- визуальной проверкой.



In [ ]:
# --- placeholder (features disabled for now) ---
print('Feature engineering is disabled in this notebook stage.')


In [ ]:
# --- legacy note ---
print('Main stage: standardization + bad 360 filtering + visualization. Chains export is available in the dedicated helper cell above if needed.')


In [ ]:
# --- JSON examples for matches (prepared items) ---
def _preview_item_json(item, max_chars=1800):
    obj = {
        'event_id': item.get('event_id'),
        'event_json': item.get('event_json'),
        'sb360_json': item.get('sb360_json'),
    }
    txt = json.dumps(obj, ensure_ascii=False, indent=2)
    return txt[:max_chars] + ('...\n[truncated]' if len(txt) > max_chars else '')


def show_json_examples_for_run(run_obj, per_match=2, max_chars=1800):
    items = run_obj.get('llm_items', run_obj.get('viz_items', []))
    with360 = [x for x in items if x.get('sb360_json') is not None]
    no360 = [x for x in items if x.get('sb360_json') is None]
    match_id = run_obj.get('match_id')
    teams = run_obj.get('cfg', {}).get('teams', [])
    ref_map = run_obj.get('cfg', {}).get('ref_by_period', {})
    print('\n' + '='*100)
    print(f'MATCH {match_id} | to_speak={run_obj.get("to_speak",0)} | teams={teams} | ref_by_period={ref_map}')
    print(f'items total={len(items)} | with360={len(with360)} | without360={len(no360)}')

    print('\n-- WITH 360 examples --')
    for i, it in enumerate(with360[:per_match], start=1):
        ej = it.get('event_json') or {}
        print(f"\n[{i}] ts={ej.get('timestamp')} | type={((ej.get('type') or {}).get('name'))} | player={((ej.get('player') or {}).get('name'))}")
        print(_preview_item_json(it, max_chars=max_chars))

    print('\n-- WITHOUT 360 examples (bad filtered or no 360) --')
    for i, it in enumerate(no360[:per_match], start=1):
        ej = it.get('event_json') or {}
        print(f"\n[{i}] ts={ej.get('timestamp')} | type={((ej.get('type') or {}).get('name'))} | player={((ej.get('player') or {}).get('name'))}")
        print(_preview_item_json(it, max_chars=max_chars))


def show_json_examples_all_runs(all_runs, per_match=2, max_chars=1800, only_to_speak=True, max_matches=None):
    mids = sorted(all_runs.keys())
    shown = 0
    for mid in mids:
        run_obj = all_runs.get(mid)
        if run_obj is None:
            continue
        if only_to_speak and int(run_obj.get('to_speak', 0)) != 1:
            continue
        show_json_examples_for_run(run_obj, per_match=per_match, max_chars=max_chars)
        shown += 1
        if max_matches is not None and shown >= max_matches:
            break


In [ ]:
# --- JSON examples: executable example ---
# only_to_speak=True -> выводит только TOP15
show_json_examples_all_runs(all_runs, per_match=2, max_chars=1800, only_to_speak=True, max_matches=None)


In [ ]:
# --- save standardized JSON artifacts (all + to_speak split) ---
def save_standardized_artifacts_for_run(run_obj, out_dir=OUT_DIR / 'processed_json'):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    mid = run_obj['match_id']
    pref = out_dir / str(mid)

    paths = {
        'events_std': Path(f"{pref}_events_std.json"),
        'events_std_clean': Path(f"{pref}_events_std_clean.json"),
        'sb360_std': Path(f"{pref}_sb360_std.json"),
        'bad_ids': Path(f"{pref}_bad_ids.json"),
        'llm_items_jsonl': Path(f"{pref}_llm_items.jsonl"),
        'meta': Path(f"{pref}_meta.json"),
    }

    paths['events_std'].write_text(json.dumps(run_obj['events_std'], ensure_ascii=False), encoding='utf-8')
    paths['events_std_clean'].write_text(json.dumps(run_obj['events_std_clean'], ensure_ascii=False), encoding='utf-8')
    paths['sb360_std'].write_text(json.dumps(run_obj['sb360_std'], ensure_ascii=False), encoding='utf-8')
    paths['bad_ids'].write_text(json.dumps(sorted(list(run_obj['bad_ids'])), ensure_ascii=False, indent=2), encoding='utf-8')

    with paths['llm_items_jsonl'].open('w', encoding='utf-8') as f:
        for row in run_obj['llm_items']:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

    meta = {
        'match_id': mid,
        'to_speak': int(run_obj.get('to_speak', 0)),
        'teams': run_obj['cfg'].get('teams', []),
        'team_colors': run_obj['cfg'].get('team_colors', {}),
        'ref_by_period': run_obj['cfg'].get('ref_by_period', {}),
        'events_raw_n': run_obj.get('events_raw_n'),
        'events_std_n': len(run_obj.get('events_std', [])),
        'events_std_clean_n': len(run_obj.get('events_std_clean', [])),
        'sb360_std_n': len(run_obj.get('sb360_std', [])),
        'bad_ids_n': len(run_obj.get('bad_ids', set())),
        'bad_reasons': run_obj.get('bad_meta', {}).get('reasons', {}),
    }
    paths['meta'].write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f"saved artifacts for {mid} -> {out_dir}")
    return paths


def save_standardized_artifacts_all_runs(all_runs, out_dir=OUT_DIR / 'processed_json'):
    rows = []
    saved = {}
    for mid, run_obj in all_runs.items():
        paths = save_standardized_artifacts_for_run(run_obj, out_dir=out_dir)
        saved[mid] = paths
        rows.append({
            'match_id': mid,
            'to_speak': int(run_obj.get('to_speak', 0)),
            'events_std': str(paths['events_std']),
            'events_std_clean': str(paths['events_std_clean']),
            'sb360_std': str(paths['sb360_std']),
            'bad_ids': str(paths['bad_ids']),
            'llm_items_jsonl': str(paths['llm_items_jsonl']),
            'meta': str(paths['meta']),
        })

    df = pd.DataFrame(rows).sort_values(['to_speak', 'match_id'], ascending=[False, True]).reset_index(drop=True)
    out_dir = Path(out_dir)
    index_all_path = out_dir / 'index_all.csv'
    df.to_csv(index_all_path, index=False)
    print('saved index:', index_all_path)

    df_to_speak = df[df['to_speak'] == 1].copy()
    index_to_speak_path = out_dir / 'index_to_speak_top15.csv'
    df_to_speak.to_csv(index_to_speak_path, index=False)
    print('saved index:', index_to_speak_path)

    return saved, df, df_to_speak


In [ ]:
# --- save standardized JSON artifacts: executable example ---
saved_artifacts, df_artifacts_all, df_artifacts_top15 = save_standardized_artifacts_all_runs(all_runs)
display(df_artifacts_all)
print('\nTOP15 to_speak artifacts:')
display(df_artifacts_top15)
